# 04 Daily Data Validation
Production quality gate for `PL_Uber_Daily_Load`.

Run this notebook **after Gold_Layer**. Any critical check that returns a non-zero failure count raises a SQL error, causing the notebook and pipeline to fail.


In [1]:
SET NOCOUNT ON;

DECLARE @Failures TABLE (
    CheckName VARCHAR(150),
    FailedRows BIGINT,
    Details VARCHAR(500)
);

/* 1. Fact grain: one row per RideID */
INSERT INTO @Failures
SELECT 'Duplicate RideID in fact_rides', COUNT_BIG(*), 'Each RideID must occur once in Gold fact.'
FROM (SELECT RideID FROM dbo.fact_rides GROUP BY RideID HAVING COUNT(*) > 1) d;

/* 2-4. SCD2 dimensions: at most one current version per business key */
INSERT INTO @Failures
SELECT 'Multiple current passenger versions', COUNT_BIG(*), 'PassengerID must have at most one IsCurrent=1 row.'
FROM (SELECT PassengerID FROM dbo.dim_passenger WHERE IsCurrent=1 GROUP BY PassengerID HAVING COUNT(*) > 1) d;

INSERT INTO @Failures
SELECT 'Multiple current driver versions', COUNT_BIG(*), 'DriverID must have at most one IsCurrent=1 row.'
FROM (SELECT DriverID FROM dbo.dim_driver WHERE IsCurrent=1 GROUP BY DriverID HAVING COUNT(*) > 1) d;

INSERT INTO @Failures
SELECT 'Multiple current city versions', COUNT_BIG(*), 'CityID must have at most one IsCurrent=1 row.'
FROM (SELECT CityID FROM dbo.dim_city WHERE IsCurrent=1 GROUP BY CityID HAVING COUNT(*) > 1) d;

/* 5-7. SCD2 effective-date integrity */
INSERT INTO @Failures
SELECT 'Invalid passenger SCD2 dates', COUNT_BIG(*), 'Current rows require EffectiveTo NULL; expired rows require EffectiveTo > EffectiveFrom.'
FROM dbo.dim_passenger WHERE (IsCurrent=1 AND EffectiveTo IS NOT NULL) OR (IsCurrent=0 AND (EffectiveTo IS NULL OR EffectiveTo <= EffectiveFrom));

INSERT INTO @Failures
SELECT 'Invalid driver SCD2 dates', COUNT_BIG(*), 'Current rows require EffectiveTo NULL; expired rows require EffectiveTo > EffectiveFrom.'
FROM dbo.dim_driver WHERE (IsCurrent=1 AND EffectiveTo IS NOT NULL) OR (IsCurrent=0 AND (EffectiveTo IS NULL OR EffectiveTo <= EffectiveFrom));

INSERT INTO @Failures
SELECT 'Invalid city SCD2 dates', COUNT_BIG(*), 'Current rows require EffectiveTo NULL; expired rows require EffectiveTo > EffectiveFrom.'
FROM dbo.dim_city WHERE (IsCurrent=1 AND EffectiveTo IS NOT NULL) OR (IsCurrent=0 AND (EffectiveTo IS NULL OR EffectiveTo <= EffectiveFrom));

/* 8-13. Fact foreign-key/orphan checks */
INSERT INTO @Failures
SELECT 'Invalid DateKey', COUNT_BIG(*), 'Fact DateKey must exist in dim_date.'
FROM dbo.fact_rides f LEFT JOIN dbo.dim_date d ON f.DateKey=d.DateKey WHERE d.DateKey IS NULL;

INSERT INTO @Failures
SELECT 'Invalid PassengerKey', COUNT_BIG(*), 'Fact PassengerKey must exist in dim_passenger.'
FROM dbo.fact_rides f LEFT JOIN dbo.dim_passenger d ON f.PassengerKey=d.PassengerKey WHERE d.PassengerKey IS NULL;

INSERT INTO @Failures
SELECT 'Invalid DriverKey', COUNT_BIG(*), 'Fact DriverKey must exist in dim_driver.'
FROM dbo.fact_rides f LEFT JOIN dbo.dim_driver d ON f.DriverKey=d.DriverKey WHERE d.DriverKey IS NULL;

INSERT INTO @Failures
SELECT 'Invalid PickupCityKey', COUNT_BIG(*), 'Fact PickupCityKey must exist in dim_city.'
FROM dbo.fact_rides f LEFT JOIN dbo.dim_city d ON f.PickupCityKey=d.CityKey WHERE d.CityKey IS NULL;

INSERT INTO @Failures
SELECT 'Invalid DropoffCityKey', COUNT_BIG(*), 'Fact DropoffCityKey must exist in dim_city.'
FROM dbo.fact_rides f LEFT JOIN dbo.dim_city d ON f.DropoffCityKey=d.CityKey WHERE d.CityKey IS NULL;

INSERT INTO @Failures
SELECT 'Invalid VehicleTypeKey', COUNT_BIG(*), 'Fact VehicleTypeKey must exist in dim_vehicle_type.'
FROM dbo.fact_rides f LEFT JOIN dbo.dim_vehicle_type d ON f.VehicleTypeKey=d.VehicleTypeKey WHERE d.VehicleTypeKey IS NULL;

/* 14. Required fact values */
INSERT INTO @Failures
SELECT 'Missing required fact values', COUNT_BIG(*), 'RideID, DateKey, core dimension keys, and BookingTimestamp are required.'
FROM dbo.fact_rides
WHERE RideID IS NULL OR DateKey IS NULL OR PassengerKey IS NULL OR DriverKey IS NULL
   OR PickupCityKey IS NULL OR DropoffCityKey IS NULL OR VehicleTypeKey IS NULL OR BookingTimestamp IS NULL;

/* Results */
SELECT CheckName, FailedRows, Details, CASE WHEN FailedRows=0 THEN 'PASS' ELSE 'FAIL' END AS Status
FROM @Failures ORDER BY CheckName;

DECLARE @FailedCheckCount INT=(SELECT COUNT(*) FROM @Failures WHERE FailedRows>0);
IF @FailedCheckCount > 0
BEGIN
    DECLARE @Message VARCHAR(500)=CONCAT('Daily data validation failed. Failed checks: ', @FailedCheckCount, '. Review validation output.');
    THROW 51000, @Message, 1;
END;

SELECT 'PASS' AS ValidationStatus, COUNT_BIG(*) AS FactRows, COUNT_BIG(DISTINCT RideID) AS UniqueRideIDs, MAX(BookingTimestamp) AS LatestBookingTimestamp
FROM dbo.fact_rides;


DatawarehouseExecuteSqlError: TYPE 'table' is not supported. Data warehouse trace id: 38a18298-dd12-4c8f-842e-08c6a556537c

## Expected result
All checks return `PASS`, followed by `ValidationStatus = PASS`. If any critical check fails, this notebook throws an error so the scheduled pipeline is marked failed and the configured failure notification can fire.
